# Extended Kalman Filter (EKF) Results Analysis

This notebook loads the exported state and covariance matrices from the EKF simulation and compares them against the "Truth" receiver trajectory. It generates standard time-series plots to evaluate the filter's estimation errors and 3-sigma uncertainty bounds.

In [ ]:
import numpy as np
from pathlib import Path
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

### Define paths based on the project structure

In [ ]:
project_root = Path.cwd().parent
data_path = project_root / 'data'

# Paths to the specific run results
results_path = data_path / 'processed' / 'filter_results' / 'sample2'
vis_data_path = data_path / 'processed' / 'visibility' / 'sample2'

# Paths to the reference user data and clock state
rcver_path = data_path / 'raw' / 'receiver_orbits' / 'eor' / 'sample2.txt'
clock_path = data_path / 'interim' / 'clock_sim' / 'true_clock_bias_drift.npy'

### Load the user's reference trajectory and plot it. 

In [ ]:
rcver_data = np.loadtxt(rcver_path, skiprows=1, usecols=(6, 13, 14, 15, 16, 17, 18))
true_ta = rcver_data[:, 0]
true_pos = 1e3 * rcver_data[:, 1:4]
true_vel = 1e3 * rcver_data[:, 4:7]

# Load Truth Clock States
clock_bias_drift = np.load(clock_path)
true_bias = clock_bias_drift[0,:]
true_drift = clock_bias_drift[1,:]

# Plot the reference trajectory in the equatorial plane
fig, ax = plt.subplots()
ax.plot(rcver_data[:, 1], rcver_data[:, 2])

# Earth
R_earth = 6378.137  # Earth radius in km
earth = plt.Circle((0, 0), R_earth, color='green', zorder=4)
ax.add_patch(earth)
	
# Earth Text Label
# Placed at a 45-degree angle (bottom right) just outside the Earth's radius
ax.text(R_earth * 0.5, -R_earth * 1, 'Earth', 
		color='green', fontsize=14, fontweight='bold', 
		ha='left', va='top', zorder=5)

ax.set_aspect('equal')
ax.set_xlabel(r'$\mathrm{X_{ECI}}$ [km]', fontsize=15)
ax.set_ylabel(r'$\mathrm{Y_{ECI}}$ [km]', fontsize=15)
ax.grid(True, which='both', linestyle='--', lw=0.5, alpha=0.5, zorder=1)


### Load EKF results

In [ ]:
print("Loading EKF results...")
t_log_array = np.load(results_path / f"tlog.npy")
estim_states = np.load(results_path / f"states.npy")
P_matrices = np.load(results_path / f"covariances.npy")

# Load Truth Clock States
clock_bias_drift = np.load(clock_path)
true_bias = clock_bias_drift[0,:]
true_drift = clock_bias_drift[1,:]

# Convert time array to hours for plotting
t_hours = t_log_array / 3600.0

print("Loading visible satellites...")
visible_sats = np.zeros_like(t_hours)
constellations = ['GPS','Galileo','GLONASS','BDS']
for const_name in constellations:
	try:
		t_vis, vis = np.loadtxt(vis_data_path/const_name/'visible_sats.dat',skiprows=1,unpack=True)
	except:
		continue
	visible_sats += vis
t_vis_hours = t_vis / 3600

print("Data loaded successfully.")

### Calculate estimation errors

In [ ]:
# 1. 3D Position and Velocity Estimation Errors (Euclidean Norm)
pos_err = np.linalg.norm(estim_states[:, 0:3] - true_pos, axis=1)
vel_err = np.linalg.norm(estim_states[:, 3:6] - true_vel, axis=1)

# 2. Clock Estimation Errors
bias_err = estim_states[:, 6] - true_bias
drift_err = estim_states[:, 7] - true_drift

# 3. 3-Sigma Uncertainty Bounds (from the trace of the covariance blocks)
pos_std = np.sqrt(P_matrices[:, 0, 0] + P_matrices[:, 1, 1] + P_matrices[:, 2, 2])
vel_std = np.sqrt(P_matrices[:, 3, 3] + P_matrices[:, 4, 4] + P_matrices[:, 5, 5])
bias_std = np.sqrt(P_matrices[:, 6, 6])
drift_std = np.sqrt(P_matrices[:, 7, 7])

### Plot the 3D position estimation error time series

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# Position estimation error time series plot
ax.semilogy(t_hours, pos_err,lw=1,c='tab:blue')
ax.semilogy(t_hours, 3*pos_std,lw=1.5,ls='-.',c='tab:blue')
ax.set_ylabel('Position estimation error [m]', fontsize=15)
ax.set_xlabel('Time [hours]', fontsize=15)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_ylim(3e-3,1e4)
t_max = 48
ax.set_xlim(0,t_max)

# Find the indices for Perigee and Apogee
# Note: If your nu_array is in degrees, wrap it in np.deg2rad(nu_array)
true_ta_rad = np.deg2rad(true_ta)
perigee_idx, _ = find_peaks(np.cos(true_ta_rad))
apogee_idx, _ = find_peaks(-np.cos(true_ta_rad))

# Add vertical lines and text for Perigee
for idx in perigee_idx:
	t_val = t_hours[idx]
	if t_val < t_max:
		# Faint dotted line spanning the height of the plot
		ax.axvline(t_val, color='gray', linestyle=':', alpha=1, lw=1.5)
		# Rotated text placed just below the upper limit
		ax.text(t_val - 0.3, 5e3, 'Perigee', rotation=90, va='top', ha='right', 
			color='gray', fontsize=10, alpha=1)

# Add vertical lines and text for Apogee
for idx in apogee_idx:
	t_val = t_hours[idx]
	if t_val < t_max:
		ax.axvline(t_val, color='tab:red', linestyle=':', alpha=1, lw=1.5)
		ax.text(t_val - 0.3, 5e3, 'Apogee', rotation=90, va='top', ha='right', 
			color='tab:red', fontsize=10, alpha=1)

# Visible satellites plot
ax2 = ax.twinx()
ax2.plot(t_hours,visible_sats,c='tab:orange',lw=1,ls='dotted',alpha=0.8,label='Sats GPS')
ax2.set_ylabel('Visible satellites', fontsize=15)
ax2.set_ylim(0,85)

legend_elements = [
    Line2D([0], [0], color='tab:blue', lw=2, linestyle='-', label=r'$\left| \hat{\mathbf{r}} - \mathbf{r} \right|$'),
    Line2D([0], [0], color='tab:blue', lw=2, linestyle='-.', label=r'$3 \sigma_{\mathbf{r}}$'),
    Line2D([0], [0], color='tab:orange', lw=2, linestyle=':', label='Visible satellites')
]

# Place the legend above the plot area, spread across 6 columns (1 line), with no frame
ax.legend(handles=legend_elements, loc='lower center', 
          bbox_to_anchor=(0.5, 0.97), ncol=6, frameon=False, fontsize=14)

# Increased padding on the title so it clears the new 1-line legend
ax.set_title('Electric orbit raising', fontsize=16, fontweight='bold',pad=35)
plt.savefig('../images/ekf_sample.jpg',dpi=300,bbox_inches='tight')
plt.show()

### Plot the 3D velocity estimation error time series

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))
# Velocity estimation error time series plot
ax.semilogy(t_hours, vel_err,lw=1,c='tab:blue')
ax.semilogy(t_hours, 3*vel_std,lw=1.5,ls='-.',c='tab:blue')
ax.set_ylabel('Velocity estimation error [m/s]', fontsize=15)
ax.set_xlabel('Time [hours]', fontsize=15)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_ylim(5e-6,2e-1)
t_max = 48
ax.set_xlim(0,t_max)

# Find the indices for Perigee and Apogee
# Note: If your nu_array is in degrees, wrap it in np.deg2rad(nu_array)
true_ta_rad = np.deg2rad(true_ta)
perigee_idx, _ = find_peaks(np.cos(true_ta_rad))
apogee_idx, _ = find_peaks(-np.cos(true_ta_rad))

# Add vertical lines and text for Perigee
for idx in perigee_idx:
	t_val = t_hours[idx]
	if t_val < t_max:
		# Faint dotted line spanning the height of the plot
		ax.axvline(t_val, color='gray', linestyle=':', alpha=1, lw=1.5)
		# Rotated text placed just below the upper limit
		ax.text(t_val - 0.3, 1e-1, 'Perigeo', rotation=90, va='top', ha='right', 
			color='gray', fontsize=10, alpha=1)

# Add vertical lines and text for Apogee
for idx in apogee_idx:
	t_val = t_hours[idx]
	if t_val < t_max:
		ax.axvline(t_val, color='tab:red', linestyle=':', alpha=1, lw=1.5)
		ax.text(t_val - 0.3, 1e-1, 'Apogeo', rotation=90, va='top', ha='right', 
			color='tab:red', fontsize=10, alpha=1)

# Visible satellites plot
ax2 = ax.twinx()
ax2.plot(t_hours,visible_sats,c='tab:orange',lw=1,ls='dotted',alpha=0.8,label='Sats GPS')
ax2.set_ylabel('Visible satellites', fontsize=15)
ax2.set_ylim(0,85)

legend_elements = [
    Line2D([0], [0], color='tab:blue', lw=2, linestyle='-', label=r'$\left| \hat{\mathbf{r}} - \mathbf{r} \right|$'),
    Line2D([0], [0], color='tab:blue', lw=2, linestyle='-.', label=r'$3 \sigma_{\mathbf{r}}$'),
    Line2D([0], [0], color='tab:orange', lw=2, linestyle=':', label='Visible satellites')
]

# Place the legend above the plot area
ax.legend(handles=legend_elements, loc='lower center', 
          bbox_to_anchor=(0.5, 0.97), ncol=6, frameon=False, fontsize=14)

# Increased padding on the title so it clears the new 1-line legend
ax.set_title('Electric orbit raising', fontsize=16, fontweight='bold',pad=35)
plt.show()

### Plot the clock bias and drift estimation error time series

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.semilogy(t_hours,np.abs(bias_err),lw=1)
ax.semilogy(t_hours,3*bias_std,lw=1.5,ls='-.',c='tab:blue',
			label=r'$\pm 3 \sigma_b$')
#ax.semilogy(t_hours,-3*bias_std,lw=1.5,ls='-.',c='tab:blue')

ax.set_ylabel(r'Clock bias estimation error [m]', fontsize=15)
ax.set_xlabel('Time [hours]', fontsize=15)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.legend(loc='upper right')
ax.set_xlim(0,48)
ax.set_ylim(1e-5,5e4)
ax.set_title('Electric orbit raising', fontsize=16, fontweight='bold')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
ax.semilogy(t_hours,np.abs(drift_err),lw=1)
ax.semilogy(t_hours,3*drift_std,lw=1.5,ls='-.',c='tab:blue',
			label=r'$\pm 3 \sigma_b$')

ax.set_ylabel(r'Clock drift estimation error [m/s]', fontsize=15)
ax.set_xlabel('Time [hours]', fontsize=15)
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.legend(loc='upper right')
ax.set_xlim(0,48)
ax.set_ylim(1e-6,1)
ax.set_title('Electric orbit raising', fontsize=16, fontweight='bold')
plt.show()

### Visualize the mahalanobis distance distribution as a filter consistency check

In [ ]:
import csv
from scipy.stats import chi2

# Load Mahalanobis Data
mah_file = results_path / f'mahalanobis.csv'
mah_time, mah_dof, mah_sq = [], [], []

try:
    with open(mah_file, mode='r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip header
        for row in reader:
            mah_time.append(float(row[0]) / 3600.0)
            mah_dof.append(float(row[1]))
            mah_sq.append(float(row[2]))
            
    # Calculate 95% dynamic threshold based on degrees of freedom (measurements)
    thresholds_95 = [chi2.ppf(0.95, df=m) for m in mah_dof]

    # Plot
    plt.figure(figsize=(10, 5))
    plt.semilogy(mah_time, mah_sq, '.', color='tab:blue', alpha=0.6, label=r'Mahalanobis $D^2$')
    plt.semilogy(mah_time, thresholds_95, '-', color='tab:red', drawstyle='steps-mid', label=r'95% $\chi^2$ Threshold')
    
    plt.xlabel('Time [hours]')
    plt.ylabel(r'Squared Distance $D^2$')
    plt.title('EKF Innovation Consistency Check (Mahalanobis Distance)')
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print(f"Mahalanobis file not found at {mah_file}. Run the filter to generate it.")